# Part 1: pre vs post stimulus (ds002680 go/no-go)

Open-data version of the SOMA part 1 pipeline. For each recording, separately for **targets** (go) and **distractors** (no-go):
- PLV / PLI / wPLI / debiased wPLI on interhemispheric homologous pairs (F3–F4, C3–C4, P3–P4, O1–O2),
- inter-trial phase coherence (Rayleigh Z) per channel, left and right hemisphere sets,

in equal-length pre, early and late windows, alpha and beta.

One recording at a time: load → filter/re-reference → epoch → compute → save that recording's rows → next.

**Note:** go trials include a button release (~330 ms), so the target `early` window contains motor activity that distractors do not.

In [ ]:
# re-read edited .py files in src/bids_conn on every cell run
%load_ext autoreload
%autoreload 2

In [ ]:
import mne
import numpy as np
import pandas as pd

from bids_conn import find_recordings, get_paths, load_raw, make_epochs, stimulus_metadata
from bids_conn.connectivity import subsample, window_connectivity, window_itc

mne.set_log_level("WARNING")

paths = get_paths()
out_dir = paths["results"] / "part1_pre_post"      # one CSV per recording goes here
out_dir.mkdir(parents=True, exist_ok=True)
print(paths["bids_root"], paths["bids_root"].exists())

## Config
Every analysis decision lives here.

In [ ]:
recs = find_recordings(paths["bids_root"], task="gonogo")
display(recs[["subject", "session", "run", "rec_id"]])
rec_ids = list(recs["rec_id"])        # all recordings; or e.g. rec_ids = rec_ids[:1] to test

# preprocessing (continuous data, before epoching)
l_freq, h_freq = 1.0, 40.0
reference      = 'average'            # original 'common' reference sits near C3/C4 and inflates their coupling
resample_to    = 500

# epochs
epoch_tmin, epoch_tmax = -0.6, 1.0    # stimuli are >= 1.8 s apart, so epochs never overlap
chans      = ['F3', 'F4', 'C3', 'C4', 'P3', 'P4', 'O1', 'O2']
reject_uv  = 100                      # peak-to-peak, checked on `chans` only (FP1/FP2 carry the blinks)
correct_only = True                   # targets with a response, distractors without
conditions = ['target', 'distractor']
equalize_trials = True                # same n per condition (PLV/PLI/wPLI depend on trial count)
seed = 0

# part 1 analysis
pairs     = [('F3', 'F4'), ('C3', 'C4'), ('P3', 'P4'), ('O1', 'O2')]
chan_sets = {'left': ['F3', 'C3', 'P3', 'O1'], 'right': ['F4', 'C4', 'P4', 'O2']}
windows   = {'pre': (-0.5, -0.01), 'early': (0.01, 0.5), 'late': (0.5, 0.99)}   # 246 samples each at 500 Hz
bands     = {'alpha': (8, 13), 'beta': (13, 30)}
mt_bandwidth = 4                      # +/-2 Hz, 1 taper (default would be +/-8 Hz)

con_kwargs = dict(
    method=['plv', 'pli', 'wpli', 'wpli2_debiased'],
    mode='multitaper',
    mt_bandwidth=mt_bandwidth,
    mt_adaptive=False,
    n_jobs=1,
    verbose=False,
)
overwrite = False                     # False = skip recordings that already have a results file

## Quality check on one recording
The evoked response should be flat before 0 with a visual response over O1/O2 from ~100 ms. Check how many trials survive rejection.

In [ ]:
qc = recs.iloc[0]
raw = load_raw(qc.path, l_freq=l_freq, h_freq=h_freq, sfreq=resample_to, reference=reference)
meta = stimulus_metadata(qc.path)
qc_epochs, qc_rej = make_epochs(raw, meta, tmin=epoch_tmin, tmax=epoch_tmax, reject_uv=reject_uv, reject_picks=chans)
print(qc_epochs, f"| rejected {qc_rej}")
print(qc_epochs.metadata.groupby(["condition", "correct"]).size())

evokeds = {cond: qc_epochs[cond].average(picks=chans) for cond in conditions}
mne.viz.plot_compare_evokeds(evokeds, picks=['O1', 'O2'], combine='mean', title=f'{qc.rec_id}: O1/O2')
del raw

## Pipeline for one recording
Everything large (`raw`, `epochs`) is local to the function, so it is freed when the function returns. Only the small result rows come back.

In [ ]:
def run_recording(rec):
    """Load one recording, compute connectivity and ITC per condition and window, return result rows."""
    raw = load_raw(rec.path, l_freq=l_freq, h_freq=h_freq, sfreq=resample_to, reference=reference)
    meta = stimulus_metadata(rec.path)
    epochs, n_rejected = make_epochs(raw, meta, tmin=epoch_tmin, tmax=epoch_tmax,
                                     reject_uv=reject_uv, reject_picks=chans)
    if correct_only:
        epochs = epochs[epochs.metadata["correct"].to_numpy()]
    epochs = epochs.pick(chans)

    by_cond = {cond: epochs[cond] for cond in conditions}
    if equalize_trials:
        rng = np.random.default_rng(seed)
        n_min = min(len(ep) for ep in by_cond.values())
        by_cond = {cond: subsample(ep, n_min, rng) for cond, ep in by_cond.items()}

    labels = dict(subject=rec.subject, session=rec.session, run=rec.run, rec_id=rec.rec_id, n_rejected=n_rejected)
    rows = []
    for cond, ep in by_cond.items():
        for row in window_connectivity(ep, windows, pairs, bands, con_kwargs):
            row.update(analysis='connectivity', condition=cond, n_trials=len(ep), **labels)
            rows.append(row)

        for set_name, set_chans in chan_sets.items():
            for row in window_itc(ep.copy().pick(set_chans), windows, bands, bandwidth=mt_bandwidth):
                row.update(analysis='itc', chan_set=set_name, condition=cond, n_trials=len(ep), **labels)
                rows.append(row)
    return rows

## Run all recordings
Each recording's rows are saved as soon as it finishes; a failing recording is reported and skipped.

**If you change the config**, set `overwrite = True` (or delete the results folder), otherwise old results are reused.

In [ ]:
failed = []
todo = recs[recs["rec_id"].isin(rec_ids)]
for i, rec in enumerate(todo.itertuples(), start=1):
    out_file = out_dir / f"{rec.rec_id}.csv"
    if out_file.exists() and not overwrite:
        print(f"[{i}/{len(todo)}] {rec.rec_id}: already done, skipping")
        continue

    try:
        rows = run_recording(rec)
    except Exception as e:
        print(f"[{i}/{len(todo)}] {rec.rec_id}: FAILED - {type(e).__name__}: {e}")
        failed.append(rec.rec_id)
        continue

    pd.DataFrame(rows).to_csv(out_file, index=False)
    print(f"[{i}/{len(todo)}] {rec.rec_id}: done, {len(rows)} rows")

print("failed:", failed)

## Combine and summarise
Runs are averaged within subject first, so every subject counts once.

In [ ]:
df = pd.concat([pd.read_csv(out_dir / f"{r}.csv", dtype={"subject": str, "session": str, "run": str})
                for r in rec_ids if (out_dir / f"{r}.csv").exists()], ignore_index=True)
df.to_csv(paths["results"] / "part1_pre_post_all_recordings.csv", index=False)
print(df.shape, "| recordings:", df["rec_id"].nunique(), "| subjects:", df["subject"].nunique())

con_df = df[df["analysis"] == "connectivity"]
con_subj = con_df.groupby(["subject", "condition", "method", "band", "ch1", "ch2", "window"], as_index=False)["value"].mean()
itc_df = df[df["analysis"] == "itc"]
itc_subj = itc_df.groupby(["subject", "condition", "band", "chan_set", "channel", "window"], as_index=False)["rayleigh_z"].mean()

In [ ]:
# connectivity: mean over subjects per condition, method, band and pair
con_subj.pivot_table(index=["method", "condition", "band", "ch1", "ch2"], columns="window", values="value")[list(windows)].round(3)

In [ ]:
# ITC: mean Rayleigh Z over subjects (~1 = no phase locking)
itc_subj.pivot_table(index=["band", "condition", "chan_set", "channel"], columns="window", values="rayleigh_z")[list(windows)].round(1)